<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 벡터와 수학 계산을 위한 NumPy 라이브러리
import numpy as np


# 두 벡터의 코사인 유사도를 계산하는 함수
def cosine(a, b):

    # 입력값을 NumPy 실수형 배열로 변환
    a = np.asarray(a, float)
    b = np.asarray(b, float)

    # 코사인 유사도 계산
    # 값이 1에 가까울수록 두 벡터의 방향이 비슷함
    return float(
        np.dot(a, b) /
        (np.linalg.norm(a) * np.linalg.norm(b))
    )


# 사용할 단어 목록
vocab = ["고양이", "강아지", "자동차"]


# 원-핫 벡터 생성
# 각 단어는 서로 다른 위치만 1이고 나머지는 0
onehot = np.eye(len(vocab))

# 고양이와 강아지의 원-핫 벡터 유사도 계산
print(cosine(onehot[0], onehot[1]))


# 각 단어의 의미를 간단한 3차원 벡터로 표현한 예제
emb = {
    "고양이": [0.90, 0.80, 0.10],
    "강아지": [0.85, 0.90, 0.15],
    "자동차": [0.10, 0.05, 0.95]
}


# 고양이와 강아지의 임베딩 벡터 유사도 계산
print(cosine(emb["고양이"], emb["강아지"]))

In [ ]:
# V : 전체 단어(토큰)의 개수
# D : 임베딩 벡터의 차원
# bytes_per_float : 실수 1개(float32)를 저장하는 데 필요한 바이트
V = 200000
D = 1024
bytes_per_float = 4


# 원-핫 벡터를 V × V 행렬로 저장한다고 가정했을 때 필요한 메모리
# 200,000개의 단어 × 200,000차원
# 결과를 GiB 단위로 변환
onehot_identity_gib = (
    V * V * bytes_per_float / 1024**3
)


# 임베딩 테이블을 V × D 행렬로 저장할 때 필요한 메모리
# 200,000개의 단어 × 1,024차원
# 결과를 GiB 단위로 변환
embedding_gib = (
    V * D * bytes_per_float / 1024**3
)


# 원-핫 행렬의 메모리 크기 출력
print(f"Dense VxV identity: {onehot_identity_gib:.2f} GiB")

# 임베딩 테이블의 메모리 크기 출력
print(f"Embedding table: {embedding_gib:.2f} GiB")

# 임베딩 테이블에 들어 있는 전체 파라미터 수 출력
print(f"Embedding parameter 수: {V * D:,}")

In [ ]:
# gensim 라이브러리 설치
!pip install gensim

In [ ]:
# Word2Vec 모델 불러오기
from gensim.models import Word2Vec


# Word2Vec 학습에 사용할 간단한 문장들
# split()을 사용하여 문장을 단어 단위로 나눔
corpus = [
    "왕 은 나라 를 다스린다".split(),
    "여왕 은 나라 를 다스린다".split(),
    "고양이 는 귀여운 동물 이다".split(),
    "자동차 는 도로 를 달린다".split(),
    "사과 는 맛있는 과일 이다".split(),
    "포도 는 달콤한 과일 이다".split(),
    "강아지 는 사람 의 친구 이다".split(),
] * 100    # 작은 데이터이므로 같은 문장을 100번 반복


# Word2Vec 모델 학습
model = Word2Vec(
    corpus,

    vector_size=50,  # 각 단어를 50차원 벡터로 표현
    window=3,        # 주변 3개 단어까지 문맥으로 사용
    min_count=1,     # 1번 이상 등장한 모든 단어를 학습
    sg=1,            # Skip-gram 방식 사용
    negative=5,      # Negative Sampling에서 음성 샘플 5개 사용
    epochs=100       # 전체 학습 데이터를 100번 반복 학습
)


# 단어 쌍의 코사인 유사도 계산
for a, b in [
    ("사과", "포도"),
    ("고양이", "강아지"),
    ("고양이", "자동차")
]:

    # 두 단어의 임베딩 벡터 사이의 유사도 출력
    print(a, b, model.wv.similarity(a, b))

In [ ]:
# CBOW와 Skip-gram에서 공통으로 사용할 학습 설정
common = dict(
    sentences=corpus,  # Word2Vec 학습에 사용할 문장 데이터
    vector_size=50,    # 단어를 50차원 벡터로 표현
    window=3,          # 주변 3개 단어까지 문맥으로 사용
    min_count=1,       # 1번 이상 등장한 모든 단어를 학습
    negative=5,        # Negative Sampling에서 음성 샘플 5개 사용
    epochs=100         # 전체 데이터를 100번 반복 학습
)


# CBOW 방식으로 Word2Vec 학습
# sg=0 : CBOW
# 주변 단어들을 보고 가운데 단어를 예측
cbow = Word2Vec(**common, sg=0)


# Skip-gram 방식으로 Word2Vec 학습
# sg=1 : Skip-gram
# 가운데 단어를 보고 주변 단어들을 예측
skip = Word2Vec(**common, sg=1)


# 두 모델에서 단어 쌍의 유사도를 각각 계산
for a, b in [
    ("사과", "포도"),
    ("고양이", "강아지")
]:

    print(
        a, b,

        # CBOW로 학습한 단어 벡터의 유사도
        "CBOW=", cbow.wv.similarity(a, b),

        # Skip-gram으로 학습한 단어 벡터의 유사도
        "Skip=", skip.wv.similarity(a, b)
    )